In [1]:
import os
import shutil

# Check if bedtools is available
if shutil.which("bedtools") is None:
    # Try to find bedtools in common conda locations
    conda_prefix = os.environ.get("CONDA_PREFIX", "")
    possible_paths = [
        f"{conda_prefix}/bin",
        os.path.expanduser("~/miniconda3/bin"),
        os.path.expanduser("~/anaconda3/bin"),
        "/usr/local/bin",
    ]
    for path in possible_paths:
        bedtools_path = os.path.join(path, "bedtools")
        if os.path.exists(bedtools_path):
            os.environ["PATH"] = path + ":" + os.environ["PATH"]
            print(f"Added {path} to PATH")
            break
    
# Verify bedtools is now available
!which bedtools && bedtools --version

In [2]:
import hummuspy.loader  # HuMMuS multilayer network utilities
import circe as ci       # ATAC-seq co-accessibility networks
import recon             # ReCoN GRN inference
import recon.infer_grn   # ReCoN GRN inference

In [5]:
import muon as mu   # Multi-omics data handling
import scanpy as sc # Single-cell analysis

/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [6]:
mudata = mu.read_h5mu("./data/build_grn_tuto/pbmc10x.h5mu")
rna = mudata.mod['rna'][:5000, :1000]   # First 1000 genes

/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [13]:
df = rna.to_df()
print(df.head())

                       LINC01409  FAM87B  LINC01128  LINC00115  FAM41C  \
smpl_AAACAGCCAATCCCTT        0.0     0.0        0.0        0.0     0.0   
smpl_AAACAGCCAATGCGCT        0.0     0.0        0.0        0.0     0.0   
smpl_AAACAGCCACCAACCG        0.0     0.0        0.0        0.0     0.0   
smpl_AAACAGCCAGGATAAC        0.0     0.0        0.0        0.0     0.0   
smpl_AAACAGCCAGTTTACG        0.0     0.0        0.0        0.0     0.0   

                       SAMD11  NOC2L  KLHL17  PLEKHN1  HES4  ...  ATP1A1-AS1  \
smpl_AAACAGCCAATCCCTT     0.0    0.0     0.0      0.0   0.0  ...         0.0   
smpl_AAACAGCCAATGCGCT     0.0    0.0     0.0      0.0   0.0  ...         0.0   
smpl_AAACAGCCACCAACCG     0.0    0.0     0.0      0.0   0.0  ...         0.0   
smpl_AAACAGCCAGGATAAC     0.0    0.0     0.0      0.0   0.0  ...         0.0   
smpl_AAACAGCCAGTTTACG     0.0    0.0     0.0      0.0   0.0  ...         0.0   

                       LINC01762  CD58  IGSF3  CD2  PTGFRN  CD101  TTF2  \

In [7]:
tfs_list = hummuspy.loader.load_tfs("human_tfs_r_hummus")

In [8]:
rna_network = recon.infer_grn.compute_rna_network(rna, tf_names=tfs_list, n_cpu=5)
print(f"RNA network: {len(rna_network)} edges")

Calculating TF-to-gene importance


Running using 5 cores: 100%|██████████| 1000/1000 [00:05<00:00, 175.55it/s]


RNA network: 25881 edges


In [9]:
print(rna_network.head(5))

    source        target     weight
17  HIVEP3     LINC02810  23.286292
12    MTF1  ARHGAP29-AS1  22.319459
24     JUN         SGIP1  21.659802
7      ID3        CELA3A  21.088190
24     JUN     LINC02238  20.940928


In [14]:
# Save RNA GRN 
rna_network.to_csv("RNA_grn.csv", index=False)